In [6]:
import numpy as np
import openml
import warnings

# Let's prepare y

In [7]:
dataset = openml.datasets.get_dataset(5)

X, y, categorical_indicator, attribute_names = dataset.get_data(
    target=dataset.default_target_attribute,
    dataset_format="dataframe"
)

# the line below will be deleted when data is cleared (at this moment we have columns with type category, but we want numeric only columns)
X = X.astype(float)

In [9]:
# this cell will be deleted when data is cleared
na_indices = X.isna().any(axis=1)
X = X.dropna()
X.reset_index(drop=True, inplace=True)

In [11]:
# this cell will be deleted when data is cleared
y_df = y.to_frame(name="Y_true_unobserved")
y_df = y_df[~na_indices].reset_index(drop=True)

In [15]:
y_df['Y_observed'] = y_df['Y_true_unobserved'].copy().astype(int)
y_df['S'] = 0
y_df['Missing_Y'] = 'no'

In [16]:
y_df

,Y_true_unobserved,Y_observed,S,Missing_Y
0,10,10,0,no
1,1,1,0,no
2,2,2,0,no
3,1,1,0,no
4,16,16,0,no
...,...,...,...,...
63,1,1,0,no
64,10,10,0,no
65,1,1,0,no
66,2,2,0,no


# **Missingness functions**

In [17]:
def MCAR_missingness(y, p=0.2):
    
    n_samples = len(y)
    n_missing = int(p * n_samples)
    missing_indices = np.random.choice(n_samples, n_missing, replace=False)
    y_missing = y.copy()
    
    y_missing.loc[missing_indices, 'Y_observed'] = -1
    y_missing.loc[missing_indices, 'S'] = 1
    y_missing.loc[missing_indices, 'Missing_Y'] = 'yes'

    return y_missing

def MAR1_missingness(y, X, feature_column, w=1, b=0):
    
    y_missing = y.copy()

    feature_values = X[feature_column].values
    z = w * feature_values + b 
    p_missing = 1 / (1 + np.exp(-z))
    
    random_values = np.random.rand(len(y))
    mask = random_values < p_missing
    
    y_missing.loc[mask, 'Y_observed'] = -1
    y_missing.loc[mask, 'S'] = 1
    y_missing.loc[mask, 'Missing_Y'] = 'yes'
    
    return y_missing

def MAR2_missingness(y, X, W=None, b=0):
    
    y_missing = y.copy()
    
    if W is None:
        W = np.random.randn(X.shape[1]) * 0.01
        
    z = np.dot(X.values, W) + b
    p_missing = 1 / (1 + np.exp(-z))
    
    random_values = np.random.rand(len(y))
    mask = random_values < p_missing
    
    y_missing.loc[mask, 'Y_observed'] = -1
    y_missing.loc[mask, 'S'] = 1
    y_missing.loc[mask, 'Missing_Y'] = 'yes'
    
    return y_missing
    
def MNAR_missingness(y, X, w_x=None, w_y=1.0, b=0.0):
    y_missing = y.copy()

    if w_x is None:
        w_x = np.random.randn(X.shape[1]) * 0.01 
        
    y_true = y_missing['Y_true_unobserved'].astype(int).values
    
    z = np.dot(X.values, w_x) + (w_y * y_true) + b
    p_missing = 1 / (1 + np.exp(-z))
    
    random_values = np.random.rand(len(y))
    mask = random_values < p_missing
    
    y_missing.loc[mask, 'Y_observed'] = -1
    y_missing.loc[mask, 'S'] = 1
    y_missing.loc[mask, 'Missing_Y'] = 'yes'
    
    return y_missing

In [18]:
y_MCAR = MCAR_missingness(y_df, p=0.2)
missing_num = y_MCAR[y_MCAR['Missing_Y'] == 'yes'].size
print(f"Number of missing values in MCAR dataset: {missing_num}")

Number of missing values in MCAR dataset: 52


In [19]:
y_MAR1 = MAR1_missingness(y_df, X, feature_column='QRS', w=-4, b=2)
missing_num = y_MAR1[y_MAR1['Missing_Y'] == 'yes'].size
print(f"Number of missing values in MAR1 dataset: {missing_num}")

Number of missing values in MAR1 dataset: 92


In [20]:
y_MAR2 = MAR2_missingness(y_df, X)
missing_num = y_MAR2[y_MAR2['Missing_Y'] == 'yes'].size
print(f"Number of missing values in MAR2 dataset: {missing_num}")

Number of missing values in MAR2 dataset: 36


In [21]:
y_MNAR = MNAR_missingness(y_df, X, w_y = -2)
missing_num = y_MNAR[y_MNAR['Missing_Y'] == 'yes'].size
print(f"Number of missing values in MNAR dataset: {missing_num}")

Number of missing values in MNAR dataset: 148


In [22]:
def merging_data_for_fista(y, X):
    y = y['Y_observed']
    
    df = X.merge(y, left_index=True, right_index=True)
    
    return df

In [23]:
df_MCAR = merging_data_for_fista(y_MCAR, X)